# 01 Build Dataset

Build the ML-ready player-game dataset using the minimum data needed for the project.

Scope for this version:
- seasons pulled: `2023-24`, `2024-25`
- train season: `2023-24`
- eval season: `2024-25`

This reduction is intentional to keep the data pull reliable and avoid timeouts.

## Imports And Configuration

In [1]:
from pathlib import Path
import json
import time

import pandas as pd
from nba_api.stats.endpoints import playergamelogs, leaguedashplayerstats, leaguedashteamstats

ROOT = Path.cwd()
DATA_DIR = ROOT / 'data'
RAW_DIR = DATA_DIR / 'raw'
INTERMEDIATE_DIR = DATA_DIR / 'intermediate'
FINAL_DIR = DATA_DIR / 'final'
OUTPUTS_DIR = ROOT / 'outputs'
RAW_PLAYERGAMELOGS_DIR = RAW_DIR / 'playergamelogs'
RAW_TEAM_CONTEXT_DIR = RAW_DIR / 'team_context'
RAW_PLAYER_CONTEXT_DIR = RAW_DIR / 'player_context'

SEASONS = ['2023-24', '2024-25']
TRAIN_SEASONS = ['2023-24']
EVAL_SEASON = '2024-25'
FORCE_PLAYERGAMELOGS = False
FORCE_CONTEXT = False
MAX_WORKERS = 6

for path in [RAW_PLAYERGAMELOGS_DIR, RAW_TEAM_CONTEXT_DIR, RAW_PLAYER_CONTEXT_DIR, INTERMEDIATE_DIR, FINAL_DIR, OUTPUTS_DIR]:
    path.mkdir(parents=True, exist_ok=True)

## Helper Functions

In [2]:
def retry(fn, retries=4, sleep_seconds=2.0, backoff=2.0):
    last_err = None
    delay = sleep_seconds
    for attempt in range(retries):
        try:
            return fn()
        except Exception as err:
            last_err = err
            if attempt == retries - 1:
                break
            time.sleep(delay)
            delay *= backoff
    raise last_err

def parse_matchup(matchup):
    if ' vs. ' in matchup:
        _, opponent = matchup.split(' vs. ')
        return 'home', opponent
    if ' @ ' in matchup:
        _, opponent = matchup.split(' @ ')
        return 'away', opponent
    return None, None

def save_json(path, payload):
    path.write_text(json.dumps(payload, indent=2))

def fetch_player_game_logs(season, force=False):
    out_path = RAW_PLAYERGAMELOGS_DIR / f'playergamelogs_{season.replace("-", "_")}.parquet'
    if out_path.exists() and not force:
        return pd.read_parquet(out_path)
    def _fetch():
        endpoint = playergamelogs.PlayerGameLogs(season_nullable=season, season_type_nullable='Regular Season', timeout=30)
        return endpoint.player_game_logs.get_data_frame()
    df = retry(_fetch)
    df.to_parquet(out_path, index=False)
    return df

def fetch_team_context(season, force=False):
    out_path = RAW_TEAM_CONTEXT_DIR / f'team_context_{season.replace("-", "_")}.parquet'
    if out_path.exists() and not force:
        print(f'team_context season={season} source=cache')
        return pd.read_parquet(out_path)
    print(f'team_context season={season} source=api starting')
    def _fetch():
        endpoint = leaguedashteamstats.LeagueDashTeamStats(
            season=season,
            season_type_all_star='Regular Season',
            measure_type_detailed_defense='Advanced',
            per_mode_detailed='PerGame',
            timeout=30,
        )
        return endpoint.league_dash_team_stats.get_data_frame()
    df = retry(_fetch)
    df.to_parquet(out_path, index=False)
    print(f'team_context season={season} source=api done rows={len(df)}')
    return df

def fetch_player_context(season, force=False):
    out_path = RAW_PLAYER_CONTEXT_DIR / f'player_context_{season.replace("-", "_")}.parquet'
    if out_path.exists() and not force:
        print(f'player_context season={season} source=cache')
        return pd.read_parquet(out_path)
    print(f'player_context season={season} source=api starting')
    def _fetch():
        endpoint = leaguedashplayerstats.LeagueDashPlayerStats(
            season=season,
            season_type_all_star='Regular Season',
            measure_type_detailed_defense='Advanced',
            per_mode_detailed='PerGame',
            timeout=30,
        )
        return endpoint.league_dash_player_stats.get_data_frame()
    df = retry(_fetch)
    df.to_parquet(out_path, index=False)
    print(f'player_context season={season} source=api done rows={len(df)}')
    return df

def standardize_logs(raw_df):
    df = raw_df.copy()
    df['home_away'], df['opponent_team'] = zip(*df['MATCHUP'].map(parse_matchup))
    df['game_date'] = pd.to_datetime(df['GAME_DATE']).dt.normalize()
    df = df.rename(columns={
        'SEASON_YEAR': 'season',
        'PLAYER_ID': 'player_id',
        'PLAYER_NAME': 'player_name',
        'TEAM_ID': 'team_id',
        'TEAM_ABBREVIATION': 'team_abbr',
        'GAME_ID': 'game_id',
        'MIN': 'min',
        'PTS': 'pts',
        'REB': 'reb',
        'AST': 'ast',
        'FGA': 'fga',
        'FG3A': 'fg3a',
        'FTA': 'fta',
    })
    df['target_pts'] = df['pts']
    df['target_reb'] = df['reb']
    df['target_ast'] = df['ast']
    keep = ['season', 'player_id', 'player_name', 'team_id', 'team_abbr', 'game_id', 'game_date', 'home_away', 'opponent_team', 'min', 'fga', 'fg3a', 'fta', 'pts', 'reb', 'ast', 'target_pts', 'target_reb', 'target_ast']
    return df[keep].copy()

def add_history(df, cols):
    out = df.sort_values(['player_id', 'season', 'game_date', 'game_id']).copy()
    for col in cols:
        shifted = out.groupby(['player_id', 'season'])[col].shift(1)
        g = [out['player_id'], out['season']]
        out[f'season_avg_{col}'] = shifted.groupby(g).expanding().mean().reset_index(level=[0, 1], drop=True)
        out[f'l5_avg_{col}'] = shifted.groupby(g).rolling(5, min_periods=1).mean().reset_index(level=[0, 1], drop=True)
        out[f'l10_avg_{col}'] = shifted.groupby(g).rolling(10, min_periods=1).mean().reset_index(level=[0, 1], drop=True)
        out[f'post_l5_{col}'] = out.groupby(['player_id', 'season'])[col].rolling(5, min_periods=1).mean().reset_index(level=[0, 1], drop=True)
        out[f'post_l10_{col}'] = out.groupby(['player_id', 'season'])[col].rolling(10, min_periods=1).mean().reset_index(level=[0, 1], drop=True)
    return out

def fetch_context_for_seasons(seasons, force=False):
    team_frames = []
    player_frames = []
    total = len(seasons)
    INTERMEDIATE_DIR.mkdir(parents=True, exist_ok=True)
    print(f'Starting context pull for {total} seasons')
    for idx, season in enumerate(seasons, start=1):
        print(f'[{idx}/{total}] season={season} team_context=starting')
        team_df = fetch_team_context(season, force=force)
        team_df = team_df[['TEAM_ID', 'TEAM_NAME', 'PACE', 'DEF_RATING', 'OFF_RATING']].copy()
        team_df['season'] = season
        team_frames.append(team_df)
        INTERMEDIATE_DIR.mkdir(parents=True, exist_ok=True)
        pd.concat(team_frames, ignore_index=True).to_parquet(INTERMEDIATE_DIR / 'team_context_partial.parquet', index=False)
        print(f'[{idx}/{total}] season={season} team_context=done rows={len(team_df)}')
        print(f'[{idx}/{total}] season={season} player_context=starting')
        player_df = fetch_player_context(season, force=force)
        player_df = player_df[['PLAYER_ID', 'USG_PCT']].copy()
        player_df['season'] = season
        player_frames.append(player_df)
        INTERMEDIATE_DIR.mkdir(parents=True, exist_ok=True)
        pd.concat(player_frames, ignore_index=True).to_parquet(INTERMEDIATE_DIR / 'player_context_partial.parquet', index=False)
        print(f'[{idx}/{total}] season={season} player_context=done rows={len(player_df)} complete={idx/total:.1%}')
    return pd.concat(team_frames, ignore_index=True), pd.concat(player_frames, ignore_index=True)

def build_availability_features(df, player_context_lookup):
    rows = []
    latest_post = {}
    seen_rosters = {}

    for (season, team_id), team_games in df.sort_values(['game_date', 'game_id']).groupby(['season', 'team_id']):
        prior_roster = set()
        for (game_date, game_id), game_rows in team_games.groupby(['game_date', 'game_id']):
            active_players = set(game_rows['player_id'].tolist())
            out_players = prior_roster - active_players
            agg = {
                'teammates_out_count': len(out_players),
                'missing_pts_l5': 0.0,
                'missing_reb_l5': 0.0,
                'missing_ast_l5': 0.0,
                'missing_min_l5': 0.0,
                'missing_pts_l10': 0.0,
                'missing_reb_l10': 0.0,
                'missing_ast_l10': 0.0,
                'missing_min_l10': 0.0,
                'missing_usg_l5': 0.0,
                'missing_usg_l10': 0.0,
            }
            for out_player in out_players:
                hist = latest_post.get((season, out_player))
                if hist is not None:
                    agg['missing_pts_l5'] += hist['post_l5_pts']
                    agg['missing_reb_l5'] += hist['post_l5_reb']
                    agg['missing_ast_l5'] += hist['post_l5_ast']
                    agg['missing_min_l5'] += hist['post_l5_min']
                    agg['missing_pts_l10'] += hist['post_l10_pts']
                    agg['missing_reb_l10'] += hist['post_l10_reb']
                    agg['missing_ast_l10'] += hist['post_l10_ast']
                    agg['missing_min_l10'] += hist['post_l10_min']
                usg = player_context_lookup.get((season, out_player))
                if pd.notna(usg):
                    agg['missing_usg_l5'] += float(usg)
                    agg['missing_usg_l10'] += float(usg)
            for row in game_rows.itertuples(index=False):
                rows.append({'player_id': row.player_id, 'game_id': row.game_id, 'team_id': row.team_id, **agg})
            prior_roster |= active_players
            for row in game_rows.itertuples(index=False):
                latest_post[(season, row.player_id)] = {
                    'post_l5_pts': row.post_l5_pts,
                    'post_l5_reb': row.post_l5_reb,
                    'post_l5_ast': row.post_l5_ast,
                    'post_l5_min': row.post_l5_min,
                    'post_l10_pts': row.post_l10_pts,
                    'post_l10_reb': row.post_l10_reb,
                    'post_l10_ast': row.post_l10_ast,
                    'post_l10_min': row.post_l10_min,
                }
    return pd.DataFrame(rows)

## Pull Player Game Logs

In [3]:
season_frames = [fetch_player_game_logs(season, force=FORCE_PLAYERGAMELOGS) for season in SEASONS]
raw_playergamelogs = pd.concat(season_frames, ignore_index=True)
raw_playergamelogs[['SEASON_YEAR', 'GAME_ID']].groupby('SEASON_YEAR').count().rename(columns={'GAME_ID': 'rows'})

,rows
SEASON_YEAR,
2023-24,26401
2024-25,26306


## Standardize The Base Table

In [4]:
base = standardize_logs(raw_playergamelogs)
base.head()

,season,player_id,player_name,team_id,team_abbr,game_id,game_date,home_away,opponent_team,min,fga,fg3a,fta,pts,reb,ast,target_pts,target_reb,target_ast
0,2023-24,2544,LeBron James,1610612747,LAL,0022301195,2024-04-14,away,NOP,37.683333,20,2,6,28,11,17,28,11,17
1,2023-24,1641713,GG Jackson,1610612763,MEM,0022301193,2024-04-14,home,DEN,44.116667,36,13,7,44,12,1,44,12,1
2,2023-24,1630202,Payton Pritchard,1610612738,BOS,0022301186,2024-04-14,home,WAS,43.716667,21,6,4,38,9,12,38,9,12
3,2023-24,203078,Bradley Beal,1610612756,PHX,0022301194,2024-04-14,away,MIN,37.616667,21,6,2,36,6,5,36,6,5
4,2023-24,1628973,Jalen Brunson,1610612752,NYK,0022301190,2024-04-14,home,CHI,41.188333,30,4,12,40,8,7,40,8,7


## Build The Season List For Context Pulls

In [5]:
sorted(base['season'].unique().tolist())

['2023-24', '2024-25']

## Pull Opponent Team Context And Player Usage

In [6]:
team_context, player_context = fetch_context_for_seasons(SEASONS, force=FORCE_CONTEXT)
team_context.head()

Starting context pull for 2 seasons
[1/2] season=2023-24 team_context=starting
team_context season=2023-24 source=api starting
team_context season=2023-24 source=api done rows=30
[1/2] season=2023-24 team_context=done rows=30
[1/2] season=2023-24 player_context=starting
player_context season=2023-24 source=api starting
player_context season=2023-24 source=api done rows=572
[1/2] season=2023-24 player_context=done rows=572 complete=50.0%
[2/2] season=2024-25 team_context=starting
team_context season=2024-25 source=api starting
team_context season=2024-25 source=api done rows=30
[2/2] season=2024-25 team_context=done rows=30
[2/2] season=2024-25 player_context=starting
player_context season=2024-25 source=api starting
player_context season=2024-25 source=api done rows=569
[2/2] season=2024-25 player_context=done rows=569 complete=100.0%


,TEAM_ID,TEAM_NAME,PACE,DEF_RATING,OFF_RATING,season
0,1610612737,Atlanta Hawks,100.84,118.4,116.4,2023-24
1,1610612738,Boston Celtics,97.98,110.6,122.2,2023-24
2,1610612751,Brooklyn Nets,97.56,115.4,112.4,2023-24
3,1610612766,Charlotte Hornets,97.81,119.2,108.6,2023-24
4,1610612741,Chicago Bulls,96.94,115.7,114.0,2023-24


## Merge Usage And Build Historical Player Features

In [7]:
player_context_small = player_context[['season', 'PLAYER_ID', 'USG_PCT']].rename(columns={'PLAYER_ID': 'player_id', 'USG_PCT': 'season_avg_usg'})
base = base.merge(player_context_small, on=['season', 'player_id'], how='left')
base = add_history(base, ['pts', 'reb', 'ast', 'min', 'fga', 'fg3a', 'fta'])
base['l5_avg_usg'] = base['season_avg_usg']
base['l10_avg_usg'] = base['season_avg_usg']
base[['player_id', 'game_date', 'season_avg_pts', 'l5_avg_pts', 'l10_avg_pts']].head()

,player_id,game_date,season_avg_pts,l5_avg_pts,l10_avg_pts
26360,2544,2023-10-24,NaN,NaN,NaN
26050,2544,2023-10-26,21.0,21.0,21.0
25525,2544,2023-10-29,21.0,21.0,21.0
25315,2544,2023-10-30,23.0,23.0,23.0
24932,2544,2023-11-01,22.0,22.0,22.0


## Merge Opponent Pace And Defensive Rating

In [8]:
abbr_lookup = raw_playergamelogs[['SEASON_YEAR', 'TEAM_ID', 'TEAM_ABBREVIATION']].drop_duplicates().rename(columns={'SEASON_YEAR': 'season', 'TEAM_ID': 'opponent_team_id', 'TEAM_ABBREVIATION': 'opponent_team'})
abbr_lookup = abbr_lookup[['season', 'opponent_team_id', 'opponent_team']].drop_duplicates()
team_context_small = team_context[['season', 'TEAM_ID', 'PACE', 'DEF_RATING']].rename(columns={'TEAM_ID': 'opponent_team_id', 'PACE': 'opponent_pace', 'DEF_RATING': 'opponent_def_rating'})
base = base.merge(abbr_lookup, on=['season', 'opponent_team'], how='left')
base = base.merge(team_context_small, on=['season', 'opponent_team_id'], how='left')
base = base.drop(columns=['opponent_team_id'])
base[['opponent_team', 'opponent_pace', 'opponent_def_rating']].head()

,opponent_team,opponent_pace,opponent_def_rating
0,DEN,97.43,112.3
1,PHX,99.00,113.7
2,SAC,99.46,114.4
3,ORL,97.37,110.8
4,LAC,97.93,114.6


## Build Availability Features

In [9]:
player_context_lookup = {(row.season, row.PLAYER_ID): row.USG_PCT for row in player_context.itertuples(index=False)}
availability_features = build_availability_features(base, player_context_lookup)
availability_features.head()

,player_id,game_id,team_id,teammates_out_count,missing_pts_l5,missing_reb_l5,missing_ast_l5,missing_min_l5,missing_pts_l10,missing_reb_l10,missing_ast_l10,missing_min_l10,missing_usg_l5,missing_usg_l10
0,203991,0022300063,1610612737,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,203992,0022300063,1610612737,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,1627749,0022300063,1610612737,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,1629027,0022300063,1610612737,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,1629631,0022300063,1610612737,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


## Assemble And Save The Final Dataset

In [10]:
ml_ready_player_games = base.merge(availability_features, on=['player_id', 'game_id', 'team_id'], how='left')
ml_ready_player_games.to_parquet(FINAL_DIR / 'ml_ready_player_games.parquet', index=False)
ml_ready_player_games.to_csv(FINAL_DIR / 'ml_ready_player_games.csv', index=False)
team_context.to_parquet(INTERMEDIATE_DIR / 'team_context.parquet', index=False)
player_context.to_parquet(INTERMEDIATE_DIR / 'player_context.parquet', index=False)
availability_features.to_parquet(INTERMEDIATE_DIR / 'availability_features.parquet', index=False)
ml_ready_player_games.head()

,season,player_id,player_name,team_id,team_abbr,game_id,game_date,home_away,opponent_team,min,...,missing_pts_l5,missing_reb_l5,missing_ast_l5,missing_min_l5,missing_pts_l10,missing_reb_l10,missing_ast_l10,missing_min_l10,missing_usg_l5,missing_usg_l10
0,2023-24,2544,LeBron James,1610612747,LAL,0022300061,2023-10-24,away,DEN,29.010000,...,0.0,0.0,0.000000,0.000000,0.0,0.0,0.000000,0.000000,0.000,0.000
1,2023-24,2544,LeBron James,1610612747,LAL,0022300076,2023-10-26,home,PHX,35.000000,...,0.0,0.0,0.000000,2.500000,0.0,0.0,0.000000,2.500000,0.259,0.259
2,2023-24,2544,LeBron James,1610612747,LAL,0022300100,2023-10-29,away,SAC,39.083333,...,0.0,0.0,0.000000,2.500000,0.0,0.0,0.000000,2.500000,0.259,0.259
3,2023-24,2544,LeBron James,1610612747,LAL,0022300111,2023-10-30,home,ORL,32.783333,...,8.0,3.0,0.333333,17.296667,8.0,3.0,0.333333,17.296667,0.441,0.441
4,2023-24,2544,LeBron James,1610612747,LAL,0022300127,2023-11-01,home,LAC,42.483333,...,24.5,5.5,4.333333,73.386667,24.5,5.5,4.333333,73.386667,0.548,0.548


## Validation Checks

In [11]:
validation = {
    'rows': int(len(ml_ready_player_games)),
    'unique_players': int(ml_ready_player_games['player_id'].nunique()),
    'unique_games': int(ml_ready_player_games['game_id'].nunique()),
    'duplicate_player_game_rows': int(ml_ready_player_games.duplicated(['player_id', 'game_id']).sum()),
    'train_rows': int(ml_ready_player_games[ml_ready_player_games['season'].isin(TRAIN_SEASONS)].shape[0]),
    'eval_rows': int(ml_ready_player_games[ml_ready_player_games['season'] == EVAL_SEASON].shape[0]),
}
(OUTPUTS_DIR / 'dataset_summary.json').write_text(json.dumps(validation, indent=2))
pd.Series(validation)

rows                          52707
unique_players                  694
unique_games                   2460
duplicate_player_game_rows        0
train_rows                    26401
eval_rows                     26306
dtype: int64

In [12]:
ml_ready_player_games.isna().mean().sort_values(ascending=False).head(25)

l10_avg_reb        0.021648
l5_avg_reb         0.021648
season_avg_min     0.021648
l10_avg_ast        0.021648
season_avg_pts     0.021648
l5_avg_pts         0.021648
l5_avg_fg3a        0.021648
l10_avg_fg3a       0.021648
l5_avg_fta         0.021648
season_avg_fta     0.021648
l10_avg_fta        0.021648
season_avg_fg3a    0.021648
l5_avg_min         0.021648
l10_avg_min        0.021648
season_avg_ast     0.021648
l5_avg_ast         0.021648
season_avg_reb     0.021648
l10_avg_pts        0.021648
l10_avg_fga        0.021648
l5_avg_fga         0.021648
season_avg_fga     0.021648
season             0.000000
game_id            0.000000
team_abbr          0.000000
team_id            0.000000
dtype: float64